In [ ]:
import pandas as pd
from data.loaders import load_single_target_tba, load_gsk_hepg2, load_pk, load_derbyshire_hepg2
from evaluate.train import generate_repeated_5xn_splits, calc_metrics
from config import SplitType, TrainConfig
from data.preprocessing import preprocess_row, getMolDescriptors, standardize, preprocess_ray
import ray
from misc import set_seeds

set_seeds(42)

In [ ]:
df, df_classification_threshold = load_gsk_hepg2()
df = preprocess_ray(df, use_features=False)
ray.shutdown()
# df

In [ ]:
train_config = TrainConfig(split_type=SplitType.SCAFFOLD, max_epochs=30, use_feats=True)
splits = generate_repeated_5xn_splits(
    df,
    train_config.n_splits,
    train_config.split_type,
    random_state=train_config.random_seed,
)

# _ = next(splits)
_, (train_df, val_df, test_df) = next(splits)

In [ ]:
from models.deltaprop import DeltapropRef
from models.deltaprop.model import DeltaProp
from models.config import DeltapropConfig

train_split, val_split, test_split, extras = DeltapropRef.prepare_splits(
    train_df=train_df, val_df=val_df, test_df=test_df
)

model_config = DeltapropConfig(encoder_n_layers=2, candidate_size=4)
model = DeltapropRef.build(model_config=model_config, **extras)

model = model.train_func(
    train_split=train_split,
    val_split=val_split,
    train_config=train_config,
    model_config=model_config,
    df_classification_threshold=df_classification_threshold
)

In [ ]:
clf_th = model.tune_binary_classification_threshold(
        val_split=val_split,
        val_labels=val_df["bin_target"],
        train_config=train_config,
        train_split=train_split,
        train_labels=train_df["bin_target"],
        df_classification_threshold=df_classification_threshold,
    )

In [ ]:
pred_probs, preds = model.predict_func(
    binary_classification_threshold=clf_th,
    train_split=train_split,
    train_labels=train_df["bin_target"],
    test_split=test_split,
    df_classification_threshold=df_classification_threshold,
)

In [ ]:
calc_metrics(pred_probs, preds, test_df["bin_target"])

In [ ]:
from models.deltaprop import embed_all

train_embeds = embed_all(train_split, model.model)
val_embeds = embed_all(val_split, model.model)
test_embeds = embed_all(test_split, model.model, scale_X_d=True)

In [ ]:
import torch

with torch.no_grad():
    pred_probs = (
        model.model.interaction(test_embeds, train_embeds)
        .sigmoid()
        .squeeze()
        .cpu()
        .numpy()
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for z in range(0, pred_probs.shape[0], 100):
    pos_mask = train_df['cont_target'] > df_classification_threshold.th
    neg_mask = ~pos_mask
    pos_prob = pred_probs[z][pos_mask].mean()
    neg_prob = pred_probs[z][neg_mask].mean()
    alt_prob = round(float((pos_prob + neg_prob) / 2), 2)

    # if not (~test_df['bin_target'].iloc[z] and alt_prob > 0.5):
    #     continue

    Y = pred_probs[z]
    X = train_df['cont_target'].to_numpy()

    plt.scatter(X, Y)
    # for i, label in enumerate(sims[z][topk_train_cand_idxs]):
    #     plt.text(X[i], Y[i], round(label, 3), fontsize=12)
    
    plt.axvline(test_df['cont_target'].iloc[z], color="blue")
    plt.axvline(df_classification_threshold.th, color="red")
    plt.axhline(0.5, color="red")
    # plt.ylim(0, 1)
    

    plt.title(f"{test_df['cont_target'].iloc[z]}, {alt_prob}, {pos_prob}")
    plt.show()
    # break

In [ ]:
# from sklearn.metrics import average_precision_score, roc_auc_score

# scores = []
# for idx in range(train_embeds.shape[0]):
#     scores.append(
#         roc_auc_score(test_df['bin_target'], pred_probs[:, idx])
#     )

# scores = np.array(scores)
# sorted_idxs = np.argsort(scores)[::-1]
# list(zip(train_df['cont_target'].iloc[sorted_idxs].astype(float), scores[sorted_idxs]))

In [ ]:
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import numpy as np


def tanimoto_similarity_matrix(
    mols_a: list,
    mols_b: list,
    radius: int = 2,
    n_bits: int = 2048,
) -> np.ndarray:
    """
    Calculate Tanimoto similarity between two lists of RDKit molecules.

    Args:
        mols_a: First list of RDKit Mol objects.
        mols_b: Second list of RDKit Mol objects.
        radius: Morgan fingerprint radius (default: 2).
        n_bits: Number of bits in the fingerprint (default: 2048).

    Returns:
        A (len(mols_a), len(mols_b)) numpy array of Tanimoto similarities.
    """
    generator = GetMorganGenerator(radius=radius, fpSize=n_bits)

    def to_fp(mol):
        if mol is None:
            raise ValueError("Invalid molecule (None) in input list.")
        return generator.GetFingerprint(mol)

    fps_a = [to_fp(m) for m in mols_a]
    fps_b = [to_fp(m) for m in mols_b]

    similarity_matrix = np.zeros((len(fps_a), len(fps_b)))

    for i, fp_a in enumerate(fps_a):
        similarity_matrix[i] = DataStructs.BulkTanimotoSimilarity(fp_a, fps_b)

    return similarity_matrix

In [ ]:
sims = tanimoto_similarity_matrix(test_mol_ds.mols, train_mol_ds.mols)
sims_topk = np.argsort(sims, axis=-1)[:, ::-1][:, :10]

sims_topk_mask = np.zeros_like(sims, dtype=np.bool)
np.put_along_axis(sims_topk_mask, sims_topk, True, axis=-1)

In [ ]:
import torch

with torch.no_grad():
    pred_probs = (
        model.interaction(test_embeds, train_embeds)
        .sigmoid()
        .squeeze()
        .cpu()
        .numpy()
    )

In [ ]:
df_classification_threshold

In [ ]:
pos_mask = train_df["bin_target"].to_numpy()
neg_mask = ~pos_mask

In [ ]:
# pos_contrib = pred_probs[:, sims_topk_mask]
# neg_contrib = pred_probs[:, sims_topk_mask]

np.where(pos_mask & sims_topk_mask, pred_probs, 0.0)

In [ ]:
pos_contrib_mask = pos_mask & sims_topk_mask
neg_contrib_mask = neg_mask & sims_topk_mask

In [ ]:
(pred_probs * pos_contrib_mask).sum(axis=-1) / (pos_contrib_mask.sum(axis=-1) + 1e-12)

In [ ]:
pos_contrib = (pred_probs * pos_contrib_mask).sum(axis=-1) / (pos_contrib_mask.sum(axis=-1) + 1e-12)
neg_contrib = (pred_probs * neg_contrib_mask).sum(axis=-1) / (neg_contrib_mask.sum(axis=-1) + 1e-12)

pos_prob = np.nan_to_num(pos_contrib)
neg_prob = np.nan_to_num(neg_contrib)
prob = (pos_prob + neg_prob) / 2

In [ ]:
train_df['cont_target'].hist()